In [ ]:
#enable autoreload

%load_ext autoreload
%autoreload all

#common jupyter functions
from IPython.display import display, clear_output
from tqdm.notebook import tqdm

#configure matplotlib
%matplotlib widget
import matplotlib.pyplot as plt
plt.ioff()

import sys, os.path
importPath = os.path.abspath('../../common')
if not importPath in sys.path:
    sys.path.append(importPath)

In [ ]:
#define paths
evalResultsPath = "./data/eval-on-words-binary.parquet"

corpusPath = "../data/preproc/ktu/LT_HS_2026-02-09-binary/corpus.ranged.json"

In [ ]:
#load data
import polars as pl
dfEval = pl.read_parquet(evalResultsPath)
display(dfEval.head(5))

#get label distribution from the corpus
from corpus.chunkedCorpus import ChunkedCorpus
corpus = ChunkedCorpus.loadFromJson(corpusPath)
labelGrps = corpus.labelGrps
print(f"Label groups: '{labelGrps}'")

#find neutral labels
neutralLblIdxs = []
offset = 0
for grpSize in labelGrps:
    neutralLblIdxs.append(offset)
    offset += grpSize
print(f"Neutral labels are: '{neutralLblIdxs}")

In [ ]:
#f1 score for each model version

import metrics.scores as scores
import numpy as np
import matplotlib.pyplot as plt

#compute f1 score for each model version
modelVersions = dfEval.select(pl.col("model_version").unique().sort()).to_series().to_list()
numLbls = len(dfEval.select(pl.col("expected_labels")).to_series()[0])

avgModelScores = []
minModelScores = []
maxModelScores = []

for modelVersion in modelVersions:
	expected = dfEval.filter(pl.col("model_version")==modelVersion).select(pl.col("expected_labels")).to_series().to_list()
	predicted = dfEval.filter(pl.col("model_version")==modelVersion).select(pl.col("found_labels")).to_series().to_list()
	
	modelLblScores = []
	lblCnts = []
	for lblIdx in range(numLbls):
		lblExpected = scores.extractLabelPreds(expected, lblIdx)
		lblPredicted = scores.extractLabelPreds(predicted, lblIdx)

		score = scores.f1score(lblExpected, lblPredicted)

		modelLblScores.append(score)
		lblCnts.append(int(sum(lblExpected)))
	
	modelLblScores = np.array(modelLblScores)
	lblCnts = np.array(lblCnts)

	avgModelScores.append(np.average(modelLblScores, weights=lblCnts))
	minModelScores.append(modelLblScores.min())
	maxModelScores.append(modelLblScores.max())


#plot
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(np.arange(len(modelVersions)), avgModelScores, label="wavg", marker=".")
ax.plot(np.arange(len(modelVersions)), minModelScores, label="min", marker=".")
ax.plot(np.arange(len(modelVersions)), maxModelScores, label="max", marker=".")

ax.set_xticks(np.arange(len(modelVersions)))
ax.set_xticklabels(modelVersions)

ax.legend()
ax.set_ylabel("F1 score")
ax.set_xlabel("Model version")
ax.set_title("F1 score by model version")

fig.show()

In [ ]:
#f1 score for each model version, but neutral labels are excluded

import metrics.scores as scores
import numpy as np
import matplotlib.pyplot as plt

#compute f1 score for each model version
modelVersions = dfEval.select(pl.col("model_version").unique().sort()).to_series().to_list()
numLbls = len(dfEval.select(pl.col("expected_labels")).to_series()[0])

avgModelScores = []
minModelScores = []
maxModelScores = []

for modelVersion in modelVersions:
	expected = dfEval.filter(pl.col("model_version")==modelVersion).select(pl.col("expected_labels")).to_series().to_list()
	predicted = dfEval.filter(pl.col("model_version")==modelVersion).select(pl.col("found_labels")).to_series().to_list()
	
	modelLblScores = []
	lblCnts = []
	for lblIdx in range(numLbls):
		if lblIdx not in neutralLblIdxs:
			lblExpected = scores.extractLabelPreds(expected, lblIdx)
			lblPredicted = scores.extractLabelPreds(predicted, lblIdx)

			score = scores.f1score(lblExpected, lblPredicted)

			modelLblScores.append(score)
			lblCnts.append(int(sum(lblExpected)))
	
	modelLblScores = np.array(modelLblScores)
	lblCnts = np.array(lblCnts)

	avgModelScores.append(np.average(modelLblScores, weights=lblCnts))
	minModelScores.append(modelLblScores.min())
	maxModelScores.append(modelLblScores.max())


#plot
fig, ax = plt.subplots(figsize=(10, 5))

barW = 1 / (3 + 2) #1 / (numGrpBars + 2forPadding)
bar = ax.bar(np.arange(len(modelVersions)), avgModelScores, width=barW, label="wavg")
ax.bar_label(bar,[f"{x:.2f}" for x in avgModelScores], padding=3)

bar = ax.bar(np.arange(len(modelVersions))+barW, minModelScores, width=barW, label="min")
ax.bar_label(bar,[f"{x:.2f}" for x in minModelScores], padding=3)

bar = ax.bar(np.arange(len(modelVersions))+barW*2, maxModelScores, width=barW, label="max")
ax.bar_label(bar, [f"{x:.2f}" for x in maxModelScores], padding=3)

ax.set_xticks(np.arange(len(modelVersions)) + barW)
ax.set_xticklabels(modelVersions)

ax.legend()
ax.set_ylabel("F1 score")
ax.set_xlabel("Model version")
ax.set_title("F1 score by model version, neutral labels excluded")

fig.show()

#TODO: add variance, avg20%best lbls, avg20%worst lbls, or avg by first and last quartiles to the chart?